# 融资信息可用时点：存档版本、政策修订及股票日历
不训练、不检验收益。初始存档不是所有时期的最早市场发布；提前预填的政策利率需服从之后已公开、在生效前宣布的真实政策。

In [1]:
from pathlib import Path
from datetime import datetime,timezone
import os,json,hashlib
import requests
import pandas as pd
import numpy as np
import qlib
from qlib.data import D
from qlib.workflow import R
from src.data.funding_availability import read_archived_rates,funding_release_events,align_funding_information
from src.data.qlib_native import build_qlib_store_from_frame,load_qlib_feature_frame,QlibProviderConfig
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/funding_availability_20261014';plan=json.loads((card/'plan.json').read_text());rawdir=root/plan['raw_directory'];series={};manifests={};archive_bytes={}
for name in ['SOFR','IOER','IORB']:
 directory=rawdir/name;assert hashlib.sha256((directory/'manifest.json').read_bytes()).hexdigest()==plan['archive_manifest_hashes'][name];series[name],manifests[name]=read_archived_rates(directory)
 for entry in manifests[name]['entries']:archive_bytes[name+'/'+entry['path']]=(directory/entry['path']).read_bytes()
# Preserve independent dated policy evidence. It corrects an advance-filled rate, not a market outcome.
from bs4 import BeautifulSoup
policy_documents={}
for event in plan['policy_events']:
 path=rawdir/(event['observation_date']+'_policy.html')
 if not path.exists():
  response=requests.get(event['source_url'],timeout=30);response.raise_for_status();path.write_bytes(response.content)
 document=path.read_bytes();text=' '.join(BeautifulSoup(document,'html.parser').get_text(' ',strip=True).split())
 assert event['required_phrase'] in text
 policy_documents[event['observation_date']]={'url':event['source_url'],'sha256':hashlib.sha256(document).hexdigest(),'document':document}
source=root/plan['raw_source'];assert hashlib.sha256(source.read_bytes()).hexdigest()==plan['raw_source_sha256'];rates=pd.DataFrame(json.loads(source.read_text())['refRates']);official_dates=pd.to_datetime(rates.effectiveDate)
events=funding_release_events(series['SOFR'],series['IOER'],series['IORB'],official_observation_dates=official_dates,policy_events=plan['policy_events'])
assert events.known.eq(1).all();current=rates.set_index(pd.to_datetime(rates.effectiveDate)).percentRate
np.testing.assert_allclose(events.sofr_percent,current.reindex(events.observation_date),atol=1e-12)
calendar_provider=root/plan['calendar_provider'];assert hashlib.sha256((calendar_provider/'qlib_store_manifest.json').read_bytes()).hexdigest()==plan['calendar_manifest_sha256'];qlib.init(provider_uri=str(calendar_provider),region='us');calendar=pd.DatetimeIndex(D.calendar(start_time=plan['start'],end_time=plan['as_of'],freq='day'))
state=align_funding_information(events,calendar);known=state.known.eq(1);assert (state.loc[known,'available_at']<state.loc[known,'decision_at']).all()
fields=['known','spread_bp','sofr_percent','reserve_rate_percent','reserve_rate_is_iorb','observation_age_days'];frame=state[['datetime',*fields]].copy();frame['instrument']='USD_FUNDING_PIT';provider=root/plan['provider'];manifest=build_qlib_store_from_frame(frame,provider_uri=provider,freq='day',as_of=plan['as_of'],field_map={f:f for f in fields},source='ALFRED earliest archived observations plus dated Fed policy notes; availability after joint archive-date EOD')
manifest.update(research_information_qualified=True,production_input_qualified=False,first_market_intraday_receipt_certified=False,availability='First native09:46 strictly after max(observation_date,SOFR archive date,policy public/archive date)23:59:59 NY');atomic_json(manifest,provider/'qlib_store_manifest.json')
native=load_qlib_feature_frame(QlibProviderConfig(provider),instruments=['USD_FUNDING_PIT'],fields=['$'+f for f in fields],start=plan['start'],end=plan['as_of']).xs('USD_FUNDING_PIT',level='instrument');np.testing.assert_allclose(native.to_numpy(),state[fields].to_numpy(),rtol=2e-7,atol=1e-4,equal_nan=True)
summary={'experiment':card.name,'decision':'conservative_archive_date_funding_information_qualified_for_research_only','new_fits':0,'new_backtests':0,'outcome_correlations':0,'objective_achieved':False,'initial_archive_rows':{k:len(v) for k,v in series.items()},'sofr_unknown_archive_rows':int(series['SOFR'].value.isna().sum()),'first_sofr_observation_in_archive':str(events.observation_date.min().date()),'first_sofr_archive_date':str(events.sofr_archive_date.min().date()),'paired_events':len(events),'native_calendar_rows':len(state),'known_rows':int(known.sum()),'first_known_decision':str(state.loc[known,'decision_at'].iloc[0]),'dated_policy_corrections':len(plan['policy_events']),'research_information_qualified':True,'production_input_qualified':False,'intraday_publication_receipt_certified':False,'provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/data.html','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/events','analysis/state']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='archive_based_funding_clock',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0,availability='joint archive dates EOD then next native09:46')
 rec.save_objects(artifact_path='inputs',plan=plan,archive_manifests=manifests,archive_bytes=archive_bytes,policy_documents=policy_documents,provider_manifest=manifest)
 rec.save_objects(artifact_path='analysis',events=events,state=state,summary=summary)
 rec.save_objects(artifact_path='sources',availability=(root/'src/data/funding_availability.py').read_text(),download=(root/'src/data/alfred_download.py').read_text())
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')),known_rows=int(known.sum()))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json')
events.loc[events.policy_source.ne('ALFRED initial')].to_csv(card/'policy_reconciliation.csv',index=False);state.loc[state.datetime.isin(pd.to_datetime(['2019-04-01','2019-04-02','2020-03-17','2020-03-18','2021-08-02','2026-10-12','2026-10-13']))].to_csv(card/'clock_examples.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'recorder_uri':str(tracking),'research_information_qualified':True,'production_input_qualified':False,'artifact_policy':summary['artifact_policy']},sort_keys=False));print({k:summary[k] for k in ['paired_events','native_calendar_rows','known_rows','first_known_decision']})


# 融资信息保守时点：使用存档与真实政策版本，不回填早期

本轮0拟合、0回测、0收益相关检验。通过官方匿名ALFRED下载表单分批取得SOFR、IOER、IORB的Initial Release Only ZIP，逐份核对README、系列名、列结构、日期范围与SHA256；原ZIP及政策文件保存NAS。SOFR1966行中84行是没有官方SOFR观察的日期，保留原档，不变成0；1882个有效SOFR观察逐日与官方现行利率相符，期间无内部缺口。

ALFRED首批可识别SOFR观察为2019-03-29，存档日期2019-04-01。不能因为NYFed原始系列从2018年开始，就赋予这些旧值未经证明的历史首发资格。保守研究信息从2019-04-02 09:46纽约时间开始；2146个原生股票日历位置全部保留，1894日有已知信息，前252日保持未知。

时间合同：max(底层有效日期、SOFR存档日期、政策利率公告/存档日期)的纽约23:59:59之后，才进入原生股票交易日09:46。按日期级证据延迟，不宣称已认证最早市场发布时刻；不选择更早的滞后来提高回测。它规避同日14:30可能修订，也无需把债券/银行假日错误等同NYSE假日。观测年龄明确保存，旧值延续不伪造为新观察。

发现并修正一个重要版本陷阱：2020-03-16 IOER的初始存档在3月13日预填1.10%，但3月15日官方实施说明已公布3月16日起为0.10%。必须服从决策前已公开的后续政策，不能机械使用最初存档。修正后该日SOFR0.26%减IOER0.10%=16bp，按SOFR存档日3月17日结束后在3月18日可用。另用2021-07-28官方说明补足IORB起始2021-07-29的0.15%记录，保留制度标记及来源；没有依据收益选取修正。

例：2026-10-12与10月13日09:46仍使用10月8日观察，因为10月9日的SOFR到10月13日才存档，保守规则不允许提前放入开盘。测试另验证其直到10月14日才能进入该保守时钟。实际历史市场可能更早知道同日08:00值，本卡不冒充精确盘中首收。

决定：conservative_archive_date_funding_information_qualified_for_research_only。派生provider具备明确日期级保守研究时点，research_information_qualified=true；production_input_qualified=false、intraday_publication_receipt_certified=false。原始有效日provider仍不能直接拼入模型。SOFR分位数/成交量的季后修订统计未进入本派生信息字段。

下一步才可对固定融资压力机制做独立预登记的信息诊断，保持不拟合、不按收益调整时钟。此利差不是SOXL实际融资成本，没有任何新增策略超额证据，现役策略未改。

主要来源：[ALFRED SOFR](https://alfred.stlouisfed.org/series?seid=SOFR)、[2020年3月15日实施说明](https://www.federalreserve.gov/newsevents/pressreleases/monetary20200315a1.htm)、[2021年7月28日实施说明](https://www.federalreserve.gov/newsevents/pressreleases/monetary20210728a1.htm)、[NYFed发布和修订政策](https://www.newyorkfed.org/markets/reference-rates/additional-information-about-reference-rates)。[政策版本核对](policy_reconciliation.csv) · [时钟示例](clock_examples.csv)。

Recorder `a1c69bbbab724a33803c22aee98888d9`。
